In [1]:
import pandas as pd

# Première tentative d'appariement au niveau juridique

In [ ]:
MCO_2022 = pd.read_csv("SAE/2022/MCO_2022r.csv", sep=";", encoding='latin1', decimal=',')
HOSPIDIAG = pd.read_csv("Hospidiag/2022/hd2022.csv", sep=";", encoding='latin1', decimal=',')
MCO_2022 = MCO_2022.drop('AN', axis=1).reset_index()
HOSPIDIAG["finess"].nunique()

In [ ]:
MCO_2022_jur = MCO_2022.groupby('FI_EJ').sum(numeric_only=True).reset_index()
MCO_2022_jur["AN"] = 2022
MCO_2022_jur["FI_EJ"].nunique()

In [ ]:
HOSPIDIAG = HOSPIDIAG.rename(columns={'finess': 'FI_EJ'})
MCO_2022_full = MCO_2022_jur.merge(HOSPIDIAG, "left", "FI_EJ")
MCO_2022_full.info()
MCO_2022_full.sample(10)

# Compréhension de la construction des indicateurs au niveau juridique dans Hospidiag issus de SAE

Nous tentons de voir si les indicateurs de personnel dans Hospidiag (finess juridique) recoupent ceux de SAE agrégé par finess juridique. On prend l'exemple de la variable CI_RH1 d'Hospidiag, qui correspond aux ETP médicaux. Précisément, d'après Hospidiag, la variable CI_RH1 de 2022 correspond à : "ETP médicaux, dont Médecins (hors anesthésistes), dont Chirurgiens (hors gynécologues-obstétriciens), dont Anesthésistes, dont Gynécologues-obstétriciens" issus du fichier Q20 de SAE 2022. Par ailleurs, "depuis la SAE 2013, tous les ETP (public et privé) sont désormais estimés à partir des ETP moyens annuels rémunérés."

In [2]:
hd = pd.read_csv("Hospidiag/2022/hd2022.csv", sep=";", encoding='latin1', decimal=',')
q20 = pd.read_csv("SAE/2022/Q20_2022r.csv", sep=';', encoding='latin1', decimal='.')
q20.head()


C:\Users\roman\AppData\Local\Temp\ipykernel_25860\1768482261.py:2: DtypeWarning: Columns (0: FI, 1: FI_EJ) have mixed types. Specify dtype option on import or set low_memory=False.
  q20 = pd.read_csv("SAE/2022/Q20_2022r.csv", sep=';', encoding='latin1', decimal='.')


,BOR,AN,FI,RS,FI_EJ,PERSO,EFFSALPLH,EFFSALPLF,EFFSALPAH,EFFSALPAF,...,ETPSALF,ETP_PU,ETP_PH,ETP_AS,ETP_HU,ETP_AT,ETP_AU,EFFSAL,ETPSAL,EFFLIB
0,Q20,2022,010000024,CH DE FLEYRIAT,010780054,M1010,10.0,7.0,2.0,13.0,...,12.74,NaN,13.15,1.0,NaN,5.62,2.35,32.0,22.12,4.0
1,Q20,2022,010000024,CH DE FLEYRIAT,010780054,M1020,2.0,1.0,NaN,1.0,...,1.29,NaN,NaN,NaN,NaN,2.01,1.29,4.0,3.30,NaN
2,Q20,2022,010000024,CH DE FLEYRIAT,010780054,M1030,8.0,4.0,NaN,5.0,...,8.03,NaN,16.06,NaN,NaN,NaN,NaN,17.0,16.06,NaN
3,Q20,2022,010000024,CH DE FLEYRIAT,010780054,M1031,5.0,1.0,1.0,1.0,...,0.92,NaN,4.24,NaN,NaN,0.10,0.12,8.0,4.46,NaN
4,Q20,2022,010000024,CH DE FLEYRIAT,010780054,M1040,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.26,1.0,0.26,NaN


In [3]:
q20_med = q20[~q20["PERSO"].isin(["M3011", "M3020", "M3030", "M3012", "M3050", "M3040", "M3060", "M3070", "M9999"])] 
#sélection de tous les personnels chirurgicaux et médicaux: on exclue les personnels médicaux "autres" de Q20, 
#conformément à la description de CI_RH1 dans hospidiag
q20_med["ETPSAL"] = pd.to_numeric(q20_med["ETPSAL"])
q20_med = q20_med[["FI_EJ", "ETPSAL"]]
q20_med = q20_med.groupby("FI_EJ").sum().reset_index()
q20_med.head()

,FI_EJ,ETPSAL
0,750826307,0.09
1,920028560,3.54
2,970200168,5.85
3,970200457,2.84
4,970203766,0.00


In [4]:
hd_etp = hd[["finess","CI_RH1"]]
hd_etp.columns = ["FI_EJ", "CI_RH1"]
hd_etp.shape

(1490, 2)

In [5]:
merge = q20_med.merge(hd_etp, how="left", on ="FI_EJ")
merge.sample(10)

,FI_EJ,ETPSAL,CI_RH1
950,450001490,3.00,NaN
1385,670013754,60.47,NaN
1571,750001067,1.00,NaN
330,150780096,94.02,114.91
172,070007927,2.00,2.00
442,240000612,1.10,NaN
109,050000645,0.40,NaN
1904,910000538,3.09,NaN
477,270000086,30.81,35.88
1793,830000477,1.11,NaN


On remarque que l'agrégation n'est pas la bonne : ETPSAL et CI_RH1 ne correspondent pas (même quand CI_RH1 est bien défini). On a donc plus d'informations avec ETPSAL du fichier Q20. Cependant, on peut craindre qu'agréger ce fichier par finess juridique ne soit pas suffisant pour apparier hospidiag à Q20, dans la mesure où on ne trouve pas les mêmes résultats.

Prenons une variable "plus simple" pour tenter de voir si dans ce cas l'agrégation de Q20 par finess juridique permet bien de retrouver les mêmes valeurs que dans Hospidiag. CI_RH5 correspond aux ETP médicaux dont gynécologues obstétriciens

In [6]:
q20_gyn = q20[q20["PERSO"]=="M2050"] 
#sélection de tous les personnels de gynécologie-obstétrique
q20_gyn["ETPSAL"] = pd.to_numeric(q20_gyn["ETPSAL"])
q20_gyn = q20_gyn[["FI_EJ", "ETPSAL"]]
q20_gyn = q20_gyn.groupby("FI_EJ").sum().reset_index()
q20_gyn.head()

hd_etp_gyn = hd[["finess","CI_RH5"]]
hd_etp_gyn.columns = ["FI_EJ", "CI_RH5"]

merge_gyn = q20_gyn.merge(hd_etp_gyn, how="left", on ="FI_EJ")
merge_gyn.sample(10)

,FI_EJ,ETPSAL,CI_RH5
503,750827933,0.30,NaN
172,2B0004246,0.25,0.25
189,320780117,6.45,6.45
565,830100533,4.46,4.46
555,820004950,0.40,0.40
611,910110014,5.67,5.18
188,310788898,7.74,NaN
40,060000635,0.00,NaN
367,590782637,7.74,7.74
356,590780227,8.84,8.84


Il semblerait que CI_RH1 corresponde finalement à ETPSAL toutes catéogies de personnel confondues (perso M9999). 

In [7]:
q20_tot = q20[q20["PERSO"]=="M9999"].reset_index()
#sélection du total d'ETP
q20_tot["ETPSAL"] = pd.to_numeric(q20_tot["ETPSAL"])
q20_tot = q20_tot[["FI_EJ", "ETPSAL"]]
q20_tot = q20_tot.groupby("FI_EJ").sum().reset_index()
q20_tot.head()


,FI_EJ,ETPSAL
0,750826307,0.09
1,920028560,4.68
2,970200168,13.68
3,970200457,3.84
4,970203766,0.76


In [8]:
merge_tot = q20_tot.merge(hd_etp, how="left", on ="FI_EJ")
print(merge_tot[merge_tot["FI_EJ"]=="630780989"])
merge_tot.head()

          FI_EJ  ETPSAL  CI_RH1
1341  630780989  614.51  614.51


,FI_EJ,ETPSAL,CI_RH1
0,750826307,0.09,NaN
1,920028560,4.68,NaN
2,970200168,13.68,NaN
3,970200457,3.84,NaN
4,970203766,0.76,NaN


On conclut donc que les variables de personnel dans Hospidiag sont bien obtenues par l'agrégation au niveau du finess juridique des variables de la SAE. Cependant, les données sont plus disponibles dans SAE que dans hospidiag. 

In [12]:
hd.shape[0] == hd["finess"].nunique()

True

Le finess juridique est bien un identifiant sur lequel on pourrait apparier hospidiag à SAE, une fois seuement les agrégations faites au niveau juridique sur SAE étant donné la forme de la base SAE.